<a id="lotka-volterra"></a>

$$
\text{Lotka-Volterra 捕食者-猎物模型 (Predator-Prey Model)} \\
x(t) \text{: 猎物 (Prey) 数量, 例如兔子 / 小鱼; } y(t) \text{: 捕食者 (Predator) 数量, 例如狐狸 / 鲨鱼} \\ \quad \\
\text{推导 (每一项都是一个 "变化率" 的假设):} \\
\text{1. 没有捕食者时, 猎物按 Malthus 模型增长: } \frac{dx}{dt} = \alpha x \\
\text{2. 捕食: 单位时间内的相遇次数 } \propto x y \text{ (质量作用, 和 SIR 的感染项一样), 每次相遇猎物减少} \\
\frac{dx}{dt} = \alpha x - \beta x y \\
\text{3. 没有猎物时, 捕食者饿死, 指数衰减: } \frac{dy}{dt} = -\gamma y \\
\text{4. 捕食者的增长来自吃掉的猎物 (转化效率不同, 系数 } \delta \text{ 而不是 } \beta \text{):} \\
\frac{dy}{dt} = \delta x y - \gamma y \\ \quad \\
\alpha \text{: 猎物出生率, } \beta \text{: 捕食率, } \gamma \text{: 捕食者死亡率, } \delta \text{: 捕食者的转化率}
$$

相关模型: [Logistic 模型](logistic_growth_model.ipynb#malthus) · [SIR 模型 (同样的质量作用项)](sir_model.ipynb#sir) · 数学基础: [ODE 方程组](../math_basics/calculus.ipynb#ode-system)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

alpha, beta, gamma, delta = 1.0, 0.1, 1.5, 0.075

def lv_rhs(t, z, alpha=alpha, beta=beta, gamma=gamma, delta=delta):
    x, y = z
    return np.array([alpha * x - beta * x * y, delta * x * y - gamma * y])

def rk4(f, t0, y0, h, n_steps):
    y = np.array(y0, dtype=float)
    ys = [y.copy()]
    t = t0
    for _ in range(n_steps):
        k1 = f(t, y)
        k2 = f(t + h / 2, y + h / 2 * k1)
        k3 = f(t + h / 2, y + h / 2 * k2)
        k4 = f(t + h, y + h * k3)
        y = y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        t += h
        ys.append(y.copy())
    return np.arange(n_steps + 1) * h + t0, np.array(ys)

z0 = [10.0, 5.0]
h, n_steps = 0.01, 3000
ts, zs = rk4(lv_rhs, 0.0, z0, h, n_steps)
sol = solve_ivp(lv_rhs, [0, ts[-1]], z0, t_eval=ts, rtol=1e-11, atol=1e-11)
print(f'手写 RK4 == solve_ivp ? {np.allclose(zs.T, sol.y, rtol=1e-6, atol=1e-6)}')

plt.figure(figsize=(4, 3))
plt.plot(ts, zs[:, 0], label='prey x')
plt.plot(ts, zs[:, 1], label='predator y')
plt.xlabel('t')
plot_nicely()
plt.show()
# 捕食者的峰值总是落后于猎物的峰值

<a id="lv-equilibrium"></a>

$$
\text{平衡点 (Equilibrium): 令两个导数都为 0} \\
x(\alpha - \beta y) = 0, \quad y(\delta x - \gamma) = 0 \implies (0, 0) \text{ 或 } \left(\frac{\gamma}{\delta}, \frac{\alpha}{\beta}\right) \\ \quad \\
\text{线性化 (Linearization): 在平衡点 } z^* \text{ 附近做一阶泰勒展开, } F(z^* + u) \approx F(z^*) + J u = J u \\
\frac{du}{dt} = J u, \quad J = \begin{bmatrix} \frac{\partial \dot{x}}{\partial x} & \frac{\partial \dot{x}}{\partial y} \\ \frac{\partial \dot{y}}{\partial x} & \frac{\partial \dot{y}}{\partial y} \end{bmatrix} = \begin{bmatrix} \alpha - \beta y & -\beta x \\ \delta y & \delta x - \gamma \end{bmatrix} \\
\text{线性方程 } u' = Ju \text{ 的解由 J 的特征值决定 (沿特征向量分解, 每个分量 } \propto e^{\lambda t} \text{):} \\
\text{实部 < 0 衰减 (稳定), 实部 > 0 增长 (不稳定), 纯虚数 } \pm i\omega \text{ 振荡 (中心 Center)} \\ \quad \\
\text{1. 在 (0, 0): } J = \begin{bmatrix} \alpha & 0 \\ 0 & -\gamma \end{bmatrix} \text{, 特征值 } \alpha > 0, -\gamma < 0 \implies \text{鞍点 (Saddle), 不稳定} \\
\text{2. 在 } (\frac{\gamma}{\delta}, \frac{\alpha}{\beta}) \text{: } J = \begin{bmatrix} 0 & -\frac{\beta\gamma}{\delta} \\ \frac{\delta\alpha}{\beta} & 0 \end{bmatrix} \\
\det(J - \lambda I) = \lambda^2 + \frac{\beta\gamma}{\delta} \cdot \frac{\delta\alpha}{\beta} = \lambda^2 + \alpha\gamma = 0 \implies \lambda = \pm i\sqrt{\alpha\gamma} \\
\text{纯虚数: 平衡点附近是闭合的周期轨道 (中心), 小振幅的周期 } T \approx \frac{2\pi}{\sqrt{\alpha\gamma}} \\
\text{(线性化方程和简谐振动 } u'' = -\alpha\gamma \, u \text{ 一样)}
$$

数学基础: [Jacobian 矩阵](../math_basics/calculus.ipynb#jacobian) · [多元泰勒展开](../math_basics/calculus.ipynb#multivariate-taylor) · [特征值](../math_basics/linear_algebra.ipynb#eigen) · [简谐振动的特征方程](../math_basics/calculus.ipynb#ode-system)

In [ ]:
x_star, y_star = gamma / delta, alpha / beta
print(f'平衡点 = ({x_star:.2f}, {y_star:.2f}), 导数 = {lv_rhs(0, [x_star, y_star])}')

def numerical_jacobian(F, z, h=1e-6):
    z = np.asarray(z, dtype=float)
    J = np.zeros((2, 2))
    for j in range(2):
        e = np.zeros(2); e[j] = h
        J[:, j] = (F(0, z + e) - F(0, z - e)) / (2 * h)
    return J

for name, point in [('(0, 0)', [0.0, 0.0]), ('(gamma/delta, alpha/beta)', [x_star, y_star])]:
    J_num = numerical_jacobian(lv_rhs, point)
    x_p, y_p = point
    J_analytic = np.array([[alpha - beta * y_p, -beta * x_p], [delta * y_p, delta * x_p - gamma]])
    print(f'{name}: J 推导 == 数值? {np.allclose(J_num, J_analytic, atol=1e-6)}, 特征值 = {np.round(np.linalg.eigvals(J_analytic), 6)}')
print(f'理论特征值 = ±{np.sqrt(alpha * gamma):.6f} i')

# 周期: 从平衡点附近出发 (小振幅) 和远离平衡点出发 (大振幅) 对比
def measure_period(z_start, t_max=40):
    s = solve_ivp(lv_rhs, [0, t_max], z_start, events=lambda t, z: z[1] - y_star, rtol=1e-11, atol=1e-11, dense_output=True)
    crossings = s.t_events[0]
    # 每个周期穿过 y = y* 两次 (上升 / 下降), 隔一个取差
    return np.mean(np.diff(crossings[::2]))

print(f'小振幅周期 = {measure_period([x_star * 1.01, y_star]):.4f}, 线性化预测 2 pi / sqrt(alpha gamma) = {2 * np.pi / np.sqrt(alpha * gamma):.4f}')
print(f'大振幅周期 = {measure_period(z0):.4f} (非线性, 周期变长)')

<a id="lv-conserved"></a>

$$
\text{守恒量 (First Integral): 为什么轨道是闭合的} \\
\text{消去 t: } \frac{dy}{dx} = \frac{y(\delta x - \gamma)}{x(\alpha - \beta y)} \xrightarrow{\text{分离变量}} \frac{\alpha - \beta y}{y} dy = \frac{\delta x - \gamma}{x} dx \\
\int \left(\frac{\alpha}{y} - \beta\right) dy = \int \left(\delta - \frac{\gamma}{x}\right) dx \implies \alpha \ln y - \beta y = \delta x - \gamma \ln x + C \\
V(x, y) = \delta x - \gamma \ln x + \beta y - \alpha \ln y = \text{常数} \\ \quad \\
\text{直接验证 (多元链式法则): } \frac{dV}{dt} = \frac{\partial V}{\partial x} \dot{x} + \frac{\partial V}{\partial y} \dot{y} = \left(\delta - \frac{\gamma}{x}\right) x (\alpha - \beta y) + \left(\beta - \frac{\alpha}{y}\right) y (\delta x - \gamma) \\
= (\delta x - \gamma)(\alpha - \beta y) + (\beta y - \alpha)(\delta x - \gamma) = 0 \\
V \text{ 在平衡点取最小值 (梯度 } \nabla V = (\delta - \frac{\gamma}{x}, \beta - \frac{\alpha}{y}) = 0 \text{, Hessian = diag}(\frac{\gamma}{x^2}, \frac{\alpha}{y^2}) \text{ 正定)} \\
\text{轨道就是 V 的等高线, 围绕最小值的等高线是闭合曲线, 所以种群数量周期性变化} \\ \quad \\
\text{一个周期内的平均值: } \frac{d \ln x}{dt} = \frac{\dot{x}}{x} = \alpha - \beta y \text{, 在一个周期 T 上积分, 左边 } \ln x(T) - \ln x(0) = 0 \\
0 = \alpha T - \beta \int_0^T y \, dt \implies \bar{y} = \frac{1}{T}\int_0^T y \, dt = \frac{\alpha}{\beta} \text{, 同理 } \bar{x} = \frac{\gamma}{\delta} \text{ (平均值 = 平衡点, 和振幅无关)}
$$

数学基础: [多元链式法则](../math_basics/calculus.ipynb#partial) · [梯度](../math_basics/calculus.ipynb#gradient) · [极值的二阶条件](../math_basics/calculus.ipynb#extremum)

In [ ]:
V = lambda x, y: delta * x - gamma * np.log(x) + beta * y - alpha * np.log(y)

V_rk4 = V(zs[:, 0], zs[:, 1])
print(f'RK4 (h = {h}): V 的变化范围 = {np.ptp(V_rk4):.2e}')

# 欧拉法: V 会持续漂移, 轨道不闭合
zs_euler = [np.array(z0)]
for _ in range(n_steps):
    zs_euler.append(zs_euler[-1] + h * lv_rhs(0, zs_euler[-1]))
zs_euler = np.array(zs_euler)
V_euler = V(zs_euler[:, 0], zs_euler[:, 1])
print(f'Euler (h = {h}): V 从 {V_euler[0]:.4f} 漂移到 {V_euler[-1]:.4f}')

# 一个周期的平均值
T = measure_period(z0)
sol_T = solve_ivp(lv_rhs, [0, T], z0, rtol=1e-11, atol=1e-11, dense_output=True)
tt = np.linspace(0, T, 20001)
xx, yy = sol_T.sol(tt)
print(f'一个周期的平均: x_bar = {np.trapezoid(xx, tt) / T:.4f} (gamma/delta = {x_star:.4f}), y_bar = {np.trapezoid(yy, tt) / T:.4f} (alpha/beta = {y_star:.4f})')

xg, yg = np.meshgrid(np.linspace(2, 70, 300), np.linspace(1, 40, 300))
plt.figure(figsize=(4, 3))
plt.contour(xg, yg, V(xg, yg), levels=12, colors='gray', linewidths=0.5)
for start in [[15.0, 8.0], [10.0, 5.0], [5.0, 3.0]]:
    _, z_orbit = rk4(lv_rhs, 0.0, start, 0.01, 1200)
    plt.plot(z_orbit[:, 0], z_orbit[:, 1], label=f'start {start}')
plt.plot(zs_euler[:, 0], zs_euler[:, 1], 'k:', linewidth=0.8, label='Euler (spirals out)')
plt.plot(x_star, y_star, 'r*', markersize=10, label='equilibrium')
plt.xlabel('prey x')
plt.ylabel('predator y')
plot_nicely()
plt.show()

<a id="volterra-principle"></a>

$$
\text{应用: Volterra 原理 (第一次世界大战期间地中海渔业的问题)} \\
\text{战争期间捕鱼减少, 但渔获中鲨鱼 (捕食者) 的比例反而上升了, 为什么?} \\ \quad \\
\text{捕捞强度 } \varepsilon \text{ 对两种鱼都按比例捕捞:} \\
\frac{dx}{dt} = (\alpha - \varepsilon) x - \beta x y, \quad \frac{dy}{dt} = \delta x y - (\gamma + \varepsilon) y \\
\text{由上面的平均值公式: } \bar{x} = \frac{\gamma + \varepsilon}{\delta}, \quad \bar{y} = \frac{\alpha - \varepsilon}{\beta} \\
\text{捕捞增加 } \implies \text{猎物 (小鱼) 平均数量增加, 捕食者 (鲨鱼) 平均数量减少} \\
\text{战争时捕捞减少 } (\varepsilon \downarrow) \implies \text{鲨鱼比例上升} \\
\text{(同理: 用农药同时杀死害虫和它的天敌, 长期来看害虫的平均数量反而会增加)}
$$

In [ ]:
for eps in [0.0, 0.2, 0.5]:
    rhs_fish = lambda t, z, e=eps: np.array([(alpha - e) * z[0] - beta * z[0] * z[1], delta * z[0] * z[1] - (gamma + e) * z[1]])
    s = solve_ivp(rhs_fish, [0, 200], z0, rtol=1e-10, atol=1e-10, dense_output=True)
    tt = np.linspace(0, 200, 200001)
    xx, yy = s.sol(tt)
    print(f'捕捞 eps = {eps}: 长期平均 x = {xx.mean():.3f} (理论 {(gamma + eps) / delta:.3f}), '
          f'y = {yy.mean():.3f} (理论 {(alpha - eps) / beta:.3f}), 鲨鱼比例 = {yy.mean() / (xx.mean() + yy.mean()):.3f}')
# 200 不是整数个周期, 所以长期平均和理论值有微小差别